In [23]:
import pandas as pd
import celloracle as co

In [26]:
CELL_TYPE = "Epi_Kit+Elf5+"
TF_OF_INTEREST = "Tfap2b"

safe_name = CELL_TYPE.replace("/", "_").replace("\\", "_").replace(" ", "_")
base_results = "celloracle_results/per_celltype"

# Path to your raw links (use filtered path if you prefer)
links_path = f"{base_results}/{safe_name}_filtered.celloracle.links"

In [27]:
links = co.load_hdf5(links_path)

print(f"Clusters: {links.cluster}")
print(f"\nColumns: {links.filtered_links[links.cluster[0]].columns.tolist()}")
links.filtered_links[links.cluster[0]].head()

Clusters: ['KO_DM', 'WT_DM']

Columns: ['source', 'target', 'coef_mean', 'coef_abs', 'p', '-logp']


,source,target,coef_mean,coef_abs,p,-logp
38578,Tfap2b,Igfbp5,0.612643,0.612643,1.986446e-14,13.701923
34718,Tfap2b,Gpc6,0.606107,0.606107,1.329005e-15,14.876473
40848,Tcf7l2,Kcnd2,0.478310,0.478310,1.755148e-19,18.755686
59252,Tfap2b,Prkg1,-0.427560,0.427560,1.067925e-15,14.971459
20972,Tfap2b,Deptor,0.409937,0.409937,4.142946e-18,17.382691


In [28]:
all_tfs = set()
for cluster_name in links.cluster:
    all_tfs.update(links.filtered_links[cluster_name]["source"].unique())

if TF_OF_INTEREST in all_tfs:
    print(f"✓ '{TF_OF_INTEREST}' found as a TF source in filtered links")
else:
    print(f"⚠ '{TF_OF_INTEREST}' NOT found in filtered links. Possible matches:")
    print([tf for tf in sorted(all_tfs) if TF_OF_INTEREST.lower() in tf.lower()])

✓ 'Tfap2b' found as a TF source in filtered links


In [29]:
wt_cluster = [c for c in links.cluster if "WT" in c][0]
df_wt_raw = links.filtered_links[wt_cluster]

df_wt = (
    df_wt_raw[df_wt_raw["source"] == TF_OF_INTEREST]
    [["source", "target", "coef_mean", "coef_abs", "p"]]
    .rename(columns={"source": "TF_source", "target": "target_gene",
                     "coef_mean": "beta_coef", "coef_abs": "beta_abs", "p": "p_value"})
    .sort_values("beta_abs", ascending=False)
    .reset_index(drop=True)
)

print(f"WT ({wt_cluster}): {len(df_wt)} targets for {TF_OF_INTEREST}")
df_wt.head(20)

WT (WT_DM): 123 targets for Tfap2b


,TF_source,target_gene,beta_coef,beta_abs,p_value
0,Tfap2b,Slc5a7,0.362612,0.362612,1.025512e-15
1,Tfap2b,Gpc6,0.308217,0.308217,3.830442e-12
2,Tfap2b,Igfbp5,0.233750,0.233750,9.198150e-13
3,Tfap2b,Gna14,0.193390,0.193390,2.612042e-16
4,Tfap2b,Maml2,0.189887,0.189887,5.702106e-16
5,Tfap2b,Deptor,0.182518,0.182518,1.806034e-14
6,Tfap2b,Erbb4,0.181537,0.181537,1.909358e-15
7,Tfap2b,Slc28a3,0.168286,0.168286,2.816584e-15
8,Tfap2b,Egfr,0.168056,0.168056,4.067998e-18
9,Tfap2b,Rftn1,0.164314,0.164314,2.744116e-14


In [32]:
negative_beta_genes_wt = df_wt[df_wt['beta_coef'] < 0]
len(negative_beta_genes_wt)

23

In [30]:
ko_cluster = [c for c in links.cluster if "KO" in c][0]
df_ko_raw = links.filtered_links[ko_cluster]

df_ko = (
    df_ko_raw[df_ko_raw["source"] == TF_OF_INTEREST]
    [["source", "target", "coef_mean", "coef_abs", "p"]]
    .rename(columns={"source": "TF_source", "target": "target_gene",
                     "coef_mean": "beta_coef", "coef_abs": "beta_abs", "p": "p_value"})
    .sort_values("beta_abs", ascending=False)
    .reset_index(drop=True)
)

print(f"KO ({ko_cluster}): {len(df_ko)} targets for {TF_OF_INTEREST}")
df_ko.head(20)

KO (KO_DM): 210 targets for Tfap2b


,TF_source,target_gene,beta_coef,beta_abs,p_value
0,Tfap2b,Igfbp5,0.612643,0.612643,1.986446e-14
1,Tfap2b,Gpc6,0.606107,0.606107,1.329005e-15
2,Tfap2b,Prkg1,-0.427560,0.427560,1.067925e-15
3,Tfap2b,Deptor,0.409937,0.409937,4.142946e-18
4,Tfap2b,Slc5a7,0.405066,0.405066,9.376216e-15
5,Tfap2b,Pde7b,0.340694,0.340694,2.212388e-17
6,Tfap2b,Rspo1,-0.309986,0.309986,1.165519e-18
7,Tfap2b,Mtmr9,0.309587,0.309587,3.806169e-16
8,Tfap2b,Thsd4,0.309584,0.309584,2.641257e-14
9,Tfap2b,Erbb4,0.303254,0.303254,3.915651e-12


In [31]:
negative_beta_genes_ko = df_ko[df_ko['beta_coef'] < 0]
len(negative_beta_genes_ko)

74

### exploration

In [5]:
base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()
base_GRN.head()

,peak_id,gene_short_name,9430076c15rik,Ac002126.6,Ac012531.1,Ac226150.2,Afp,Ahr,Ahrr,Aire,...,Znf784,Znf8,Znf816,Znf85,Zscan10,Zscan16,Zscan22,Zscan26,Zscan31,Zscan4
0,chr10_100050979_100052296,4930430F08Rik,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10_101006922_101007748,SNORA17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,chr10_101144061_101145000,Mgat4c,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,chr10_10148873_10149183,9130014G24Rik,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr10_10149425_10149815,9130014G24Rik,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
tf_col = "Tfap2b"  # replace with your column of interest
genes = base_GRN.loc[base_GRN[tf_col] == 1, "gene_short_name"].tolist()

In [14]:
len(genes)

45493

In [15]:
len(base_GRN)

91976

In [17]:
import scanpy as sc
adata_full = sc.read_h5ad("CTR9_snRNASeq/CTR9_snRNASeq_full.h5ad")
print(f"Loaded data: {adata_full.shape[0]} cells x {adata_full.shape[1]} genes")
print(f"\nCell types (cluster_annot): {adata_full.obs['cluster_annot'].unique().tolist()}")
print(f"Samples: {adata_full.obs['sample'].unique().tolist()}")
adata_full

Loaded data: 9869 cells x 33696 genes

Cell types (cluster_annot): ['Epi_Kit+Elf5+', 'Epi_Ctr9+', 'Bcells', 'Pericytes/SMC', 'BasalEpi_Acta2+Trp63', 'Adipocyte', 'Endothelials', 'Fibroblasts', 'Tcells', 'DCs', 'Myeloid_cells', 'Schwann?', 'Epi_proliferating', 'SMC?']
Samples: ['WT_DM', 'KO_DM']


AnnData object with n_obs × n_vars = 9869 × 33696
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'RNA_snn_res.0.5', 'seurat_clusters', 'RNA_snn_res.0.1', 'RNA_snn_res.1', 'RNA_snn_res.0.2', 'cluster_annot'

In [19]:
genes_scrna = adata_full.var_names.tolist()
common = list(set(genes) & set(genes_scrna))
len(common)

13264